# 판촉사랑(87sarang) 구매처명 매칭 테스트 노트북

pandas + relaxed 정규화 + KR-SBERT 유사도 매칭


## 사용법

- 이 노트북은 로직 검증/디버그용입니다. 실제 운영 실행은 `02_match_buyer_names.py`를 사용하세요.
- 설정값(경로, 배정 순서, BERT 임계값 등)은 모두 `match_config.py`에서 관리합니다. 값을 바꾸려면 이 노트북이 아니라 `match_config.py`를 수정하세요.
- 로그는 `match_logging.py`를 통해 콘솔과 `output/match.log`에 함께 기록됩니다.
- BERT 임베딩은 `output/embedding_cache.pkl`에 모델별로 캐시됩니다. 같은 상품명은 재실행해도 다시 계산하지 않습니다.
- 셀은 위에서부터 순서대로 실행하세요.
- `0. BERT 사용 환경 사전 점검`과 `5-5. BERT 매칭 실행` 셀은 sentence-transformers 설치 여부/CUDA 사용 가능 여부를 먼저 확인합니다. 여기서 오류가 나면 `5-4`(raw/exact/relaxed 매칭)는 다시 돌릴 필요 없이, 환경만 고치고 `5-5` 셀만 다시 실행하면 됩니다.


## 판촉사랑 구매처 매칭 1대1 - relaxed + KR-SBERT 상품명 유사도 버전


In [ ]:
import re
import sys
import pickle
from pathlib import Path
from datetime import datetime, date

import pandas as pd

from match_config import *  # noqa: F401,F403 (설정값은 match_config.py에서 관리)
from match_logging import log

log("Python:", sys.version)
log("pandas version:", pd.__version__)


def display(obj):
    """Jupyter의 display()를 스크립트에서도 쓸 수 있게 하는 얕은 대체 함수."""
    try:
        log(obj.to_string())
    except AttributeError:
        log(obj)


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

log("sarang87 폴더:", BASE_DIR)
log("output 폴더:", OUTPUT_DIR)
log(f"{SOURCE_LABEL} 파일:", FILE_87)
log("거래데이터 파일:", FILE_TR)
log("상품분류 카테고리 파일:", FILE_CATEGORY)
log("결과 파일:", OUTPUT_FILE)

assert FILE_87.exists(), f"파일이 없습니다: {FILE_87}"
assert FILE_TR.exists(), f"파일이 없습니다: {FILE_TR}"
assert FILE_CATEGORY.exists(), f"파일이 없습니다: {FILE_CATEGORY}"

log("거래데이터 배정 순서:", TRADE_ORDER_MODE)
log("인쇄방법 조건: 제외 / 상세시트 확인용으로만 유지")
log("공통 필터 순서: 구매처분류(중) → 구매처분류(소) → 날짜")
log("상품명 비교 순서: 완전일치 → exact 기본정규화 → relaxed 보정정규화 → BERT 유사도")
log("BERT 사용:", USE_BERT_PRODUCT_SIMILARITY)
log("BERT 모델:", BERT_MODEL_NAME)
log("BERT 자동매칭 기준:", BERT_SIM_THRESHOLD)
log("BERT 후보 수 제한: 없음")
log("전체 배정 순서: 상품명완전일치 전체 → exact 전체 → relaxed 전체 → BERT 최고유사도")
log("BERT 배정 기준: 중분류+소분류+날짜 후보 중 미사용 후보의 최고 유사도 1개 배정")
log("분류 예외 7번: 거래데이터 전시회/박람회/축제/행사 중분류는 판촉사랑 기념행사별+전시회 중분류 후보로 확인")
log("분류 예외 추가: 판촉사랑 관광지/국립공원/놀이공원 하위 소분류는 거래데이터 골프관련/관광지 후보도 확인")
log("BERT_DEVICE:", BERT_DEVICE)
log("REQUIRE_CUDA_GPU:", REQUIRE_CUDA_GPU)


## 0. BERT 사용 환경 사전 점검


In [ ]:
# 실제 매칭(파일 읽기 ~ relaxed 단계)을 다 돌리고 나서 맨 마지막 BERT 단계에서야
# sentence-transformers 미설치 / CUDA 미사용 문제를 알게 되면 그 앞 단계를 처음부터 다시
# 돌려야 하므로, 무거운 모델 로딩(get_bert_model)은 그대로 지연시키되 환경 점검만 여기서
# 미리 해서 문제가 있으면 매칭을 시작하기 전에 바로 중단합니다.

def check_cuda_or_raise():
    """torch CUDA 사용 가능 여부를 확인합니다. GPU가 없으면 중단합니다."""
    try:
        import torch
    except ImportError as exc:
        raise ImportError(
            "GPU BERT를 사용하려면 torch 설치가 필요합니다. "
            "예: pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126"
        ) from exc

    log("torch version:", torch.__version__)
    log("torch CUDA build:", torch.version.cuda)
    log("cuda available:", torch.cuda.is_available())

    if not torch.cuda.is_available():
        message = (
            "CUDA GPU를 사용할 수 없습니다.\n"
            "현재 선택한 VS Code 커널이 GPU 지원 torch를 사용하지 않거나, NVIDIA 드라이버/CUDA 환경이 맞지 않습니다.\n"
            "커널이 'Python 3.11 (work GPU)' 또는 work 환경으로 잡혀 있는지 확인하세요."
        )
        if REQUIRE_CUDA_GPU:
            raise RuntimeError(message)
        log(message)
        return "cpu"

    log("GPU:", torch.cuda.get_device_name(0))
    return "cuda"


if USE_BERT_PRODUCT_SIMILARITY:
    try:
        import sentence_transformers  # noqa: F401
    except ImportError as exc:
        raise ImportError(
            "BERT 상품명 유사도를 사용하려면 sentence-transformers 설치가 필요합니다. "
            "노트북/터미널에서 `pip install sentence-transformers` 실행 후 다시 실행하세요."
        ) from exc

    if BERT_DEVICE == "cuda":
        check_cuda_or_raise()

    log("BERT 사용 환경 사전 점검 통과")


## 1. 파일 읽기


In [ ]:
def read_csv_auto(path):
    """CSV 인코딩 자동 시도. 판촉사랑 파일은 utf-8-sig가 우선입니다."""
    encodings = ["utf-8-sig", "cp949", "euc-kr", "utf-8"]
    last_error = None

    for enc in encodings:
        try:
            return pd.read_csv(path, dtype=str, encoding=enc, keep_default_na=False), enc
        except Exception as exc:
            last_error = exc

    raise RuntimeError(f"CSV 파일을 읽지 못했습니다: {path}\n마지막 오류: {last_error}")


def read_table_auto(path):
    """확장자에 따라 CSV 또는 Excel을 자동으로 읽습니다. Excel은 첫 번째 시트를 읽습니다."""
    suffix = str(path).lower()
    if suffix.endswith(".csv"):
        df, enc = read_csv_auto(path)
        log(f"{SOURCE_LABEL} CSV 인코딩:", enc)
        return df
    if suffix.endswith((".xlsx", ".xls")):
        return pd.read_excel(path, dtype=str, keep_default_na=False)
    raise ValueError(f"지원하지 않는 파일 형식입니다: {path}")


# 판촉사랑 Excel / 거래데이터 Excel 읽기
log("pandas 방식으로 파일을 읽습니다.")
# 변수명 df87은 기존 코드 재사용을 위해 그대로 둡니다.
df87 = read_table_auto(FILE_87)
dftr = pd.read_excel(FILE_TR, dtype=str, keep_default_na=False)

# 컬럼명 문자열화 및 앞뒤 공백 정리
# 거래데이터에는 '구매처 명 '처럼 뒤 공백이 있는 컬럼도 있어 원본명은 최대한 유지하되 문자열화합니다.
df87.columns = [str(c).strip() for c in df87.columns]
dftr.columns = [str(c) for c in dftr.columns]

# 상품코드는 앞자리 0이 중요하므로 문자열 상태로 유지합니다.
# 여기서는 zfill을 하지 않고, 입력 파일에 있는 상품코드를 그대로 사용합니다.
if "상품코드" in df87.columns:
    df87["상품코드"] = df87["상품코드"].map(lambda x: "" if pd.isna(x) else str(x).strip())

log(f"{SOURCE_LABEL} 행 수:", len(df87))
log("거래데이터 행 수:", len(dftr))
log(f"{SOURCE_LABEL} 컬럼:", list(df87.columns))
log("거래데이터 컬럼:", list(dftr.columns))


## 2. 정규화 함수


In [ ]:
def clean_text(value):
    """빈값/공백/NBSP 정리용."""
    if pd.isna(value):
        return ""
    text = str(value).replace("\u00a0", " ").strip()
    text = re.sub(r"\s+", " ", text)
    return text


# 상품명 앞에 붙을 수 있는 제작/상태 태그만 제거합니다.
# 브랜드 태그([PGA], [쿨린], [스위스밀리터리])는 제거하지 않습니다.
PRODUCT_PREFIX_TAGS = {
    "긴급제작", "급행제작", "빠른제작", "당일제작", "주문제작",
    "긴급", "급행", "당일", "빠른제작상품"
}

# 제작/상태 태그 비교용: 태그 안 공백 차이를 없애고 비교합니다.
# 예: "긴급 제작"과 "긴급제작"을 같은 태그로 처리하기 위한 값입니다.
PRODUCT_PREFIX_TAGS_NORM = {
    re.sub(r"\s+", "", x) for x in PRODUCT_PREFIX_TAGS
}


def normalize_product_raw(value):
    """상품명 완전일치용: 원본 상품명의 앞뒤 공백, 연속 공백만 정리."""
    return clean_text(value)


def normalize_product_exact(value):
    """상품명 exact용: 기본 정규화 후 비교.

    완전일치(raw)보다 한 단계 느슨하지만, relaxed처럼 상품명 의미 요소를 많이 지우지는 않습니다.
    처리 내용:
    - 앞뒤/연속 공백 정리
    - 영문 대소문자 통일
    - 공백 및 일부 구분자 제거
    - 괄호/대괄호 기호는 제거하되 내부 글자는 유지
    """
    text = clean_text(value).lower()
    if not text:
        return ""
    # 괄호/대괄호는 글자는 남기고 기호만 제거되도록 전체 구분자를 제거합니다.
    text = re.sub(r"[\s ]+", "", text)
    text = re.sub(r'''[·ㆍ･\-_/,.()\[\]{}<>:;|"\'`~!@#$%^&*=\\?]+''', "", text)
    return text


def normalize_product_relaxed(value):
    """
    상품명 보조매칭용.

    처리 규칙:
    1. 맨 앞 (완판), (인쇄 무료 이벤트) 같은 상태/이벤트 문구는 제거
       - 예: (완판) 상품명 → 상품명
       - 예: (인쇄 무료 이벤트) 상품명 → 상품명

    2. 맨 앞 [긴급제작], [급행제작] 같은 제작/상태 태그는 제거
       - 예: [긴급제작] 화이트 물티슈 → 화이트 물티슈

    3. 맨 뒤 [102-217591] 같은 코드성 대괄호는 제거
       - 예: 상품명 [102-217591] → 상품명

    4. [쿨린], [PGA], [LION] 같은 브랜드 태그는 []만 제거하고 글자는 유지
       - 예: [쿨린] 리풀 → 쿨린 리풀
       - 예: [LION] 참그린 → LION 참그린

    5. 상품명 안의 단독 1P / 1개 / 1EA 표기는 제거
       - 예: 12cm연필 책갈피자(칼라) 1P → 12cm연필 책갈피자(칼라)

    6. 쇼핑백 증정 문구 제거
       - 예: (쇼핑백 증정) → 제거

    7. 우산 상품의 8K / 10K / 16K 같은 살대 수 표기 제거
       - 단, 상품명에 '우산'이 있을 때만 적용

    8. 우산/양우산 표현 통일
       - 예: 양우산 → 우산

    9. + 기호 제거
       - 예: 롱티스푼+롱포크 → 롱티스푼롱포크

    10. 단독 색상명 '실버' 제거
        - 예: 실버 그레이스 롱티스푼롱포크 → 그레이스 롱티스푼롱포크
        - 단, 다른 단어 안에 붙은 경우는 제거하지 않음

    11. 상품명 앞의 (국산) 제거
        - 예: (국산)코리아5링 니들3색 터치펜 → 코리아5링 니들3색 터치펜

    12. 상품명 안의 사이즈 괄호 제거
        - 예: 포켓에코백 A형 (34x39cm) 1P → 포켓에코백 A형 1P
        - 예: 포켓에코백 A형 (34 x 39 cm) 1P → 포켓에코백 A형 1P

    13. 상품명 뒤의 잉크/저점도 설명 괄호 제거
        - 예: 코리아5링 니들3색 터치펜(독일잉크/초초저점도) → 코리아5링 니들3색 터치펜

    14. 공백 정리
    """
    text = normalize_product_raw(value)

    # 맨 앞의 상태/이벤트 괄호 문구 제거
    # 예: (완판), (품절), (인쇄 무료 이벤트)
    text = re.sub(
        r"^\((완판|품절|일시품절|단종|판매종료|인쇄\s*무료\s*이벤트)\)\s*",
        "",
        text
    )

    # 상품명 앞의 (국산) 제거
    # 예: (국산)코리아5링 니들3색 터치펜 → 코리아5링 니들3색 터치펜
    text = re.sub(r"^\(\s*국산\s*\)\s*", "", text)

    if USE_PRODUCT_PREFIX_TAG_REMOVAL:
        # 맨 앞에 제작상태 태그가 여러 개 붙은 경우도 처리
        # 예: [긴급제작] [당일제작] 상품명 → 상품명
        while True:
            m = re.match(r"^\[([^\]]+)\]\s*(.*)$", text)
            if not m:
                break

            tag = clean_text(m.group(1))
            rest = clean_text(m.group(2))
            tag_norm = re.sub(r"\s+", "", tag)

            # 제작/상태 태그면 태그 자체를 제거
            if tag_norm in PRODUCT_PREFIX_TAGS_NORM and rest:
                text = rest
            else:
                break

    # 맨 뒤에 붙은 코드성 대괄호 제거
    # 예: 상품명 [102-217591] → 상품명
    # 코드는 항상 숫자를 포함하므로, 숫자가 없는 [LION]/[PGA] 같은 브랜드 태그가
    # 이 단계에서 함께 삭제되지 않도록 대괄호 안에 숫자가 있을 때만 제거합니다.
    text = re.sub(r"\s*\[(?=[0-9A-Za-z_-]*\d)[0-9A-Za-z_-]{3,}\]\s*$", "", text)

    # 브랜드/제품 태그는 []만 제거하고 글자는 유지
    # 예: [쿨린] 리풀 → 쿨린 리풀
    # 예: [LION] 참그린 → LION 참그린
    text = re.sub(r"\[([^\]]+)\]", r" \1 ", text)

    # 상품명 안의 사이즈 괄호 제거
    # 예:
    # 포켓에코백 A형 (34x39cm) 1P → 포켓에코백 A형 1P
    # 포켓에코백 A형 (34 x 39 cm) 1P → 포켓에코백 A형 1P
    # 단, 숫자 x 숫자 형태의 크기 표기만 제거합니다.
    text = re.sub(
        r"\(\s*\d+(?:\.\d+)?\s*(?:x|X|×|\*)\s*\d+(?:\.\d+)?\s*(?:cm|CM|mm|MM|m|M)?\s*\)",
        " ",
        text
    )

    # 단독 1P / 1개 / 1EA 제거
    # 단, 10P / 2P / 10000mAh / 3in1 / 1PORT / 1+1 은 제거하지 않음
    text = re.sub(
        r"(?<![0-9A-Za-z가-힣])1\s*(?:p|P|개|ea|EA)(?![0-9A-Za-z가-힣])",
        " ",
        text
    )

    # 사은품/증정 문구 제거
    # 예: (쇼핑백 증정), 쇼핑백증정
    text = re.sub(r"\(?\s*쇼핑백\s*증정\s*\)?", " ", text)

    # 우산 상품의 살대 수 표기 제거
    # 예: 10K 3단 자동 우산 → 3단 자동 우산
    # 단, 상품명에 '우산'이 있을 때만 적용
    if "우산" in text:
        text = re.sub(
            r"(?<![0-9A-Za-z가-힣])\d{1,2}\s*[kK](?![0-9A-Za-z가-힣])",
            " ",
            text
        )

    # 우산/양우산 표현 통일
    # 예: 거꾸로 양우산 → 거꾸로 우산
    text = re.sub(r"양우산", "우산", text)

    # + 기호 제거
    # 예: 롱티스푼+롱포크 → 롱티스푼롱포크
    text = re.sub(r"\s*\+\s*", "", text)

    # 단독 색상명 '실버' 제거
    # 예: 실버 그레이스 롱티스푼롱포크 → 그레이스 롱티스푼롱포크
    text = re.sub(
        r"(?<![0-9A-Za-z가-힣])(?:실버|silver)(?![0-9A-Za-z가-힣])",
        " ",
        text,
        flags=re.IGNORECASE
    )

    # 상품명 뒤의 잉크/저점도 설명 괄호 제거
    # 예: 코리아5링 니들3색 터치펜(독일잉크/초초저점도) → 코리아5링 니들3색 터치펜
    # 괄호가 상품명 끝에 있고, 잉크/저점도 설명일 때만 제거합니다.
    text = re.sub(
        r"\([^)]*(?:잉크|저점도|초저점도|초초저점도)[^)]*\)\s*$",
        " ",
        text
    )

    # 공백 정리
    text = re.sub(r"\s+", " ", text).strip()

    return text


def normalize_category_raw(value):
    """분류명 비교용 기본 정규화.

    예:
    - 중.고등학교 / 중·고등학교 / 중고등학교 → 중고등학교로 비교 가능
    - 학교/교육기관처럼 구분자가 들어간 값은 구분자 제거 후 비교
    """
    text = clean_text(value).lower()
    text = text.replace("·", "").replace("ㆍ", "").replace("･", "")
    text = re.sub(r"[\s\t\n\r]+", "", text)
    text = re.sub(r"[\\/\-_,.()\[\]{}<>:;|]+", "", text)
    return text


# 중/소분류 별칭. 양쪽 모두 같은 canonical로 묶습니다.
CATEGORY_ALIAS_PAIRS = [
    ("교육지원청/wee센터/도서관", "교육청/wee센터/도서관"),
    ("봉사/자선단체/사회복지기금", "봉사/자선단체/사회복지기금,센터기타"),
    ("전문직(변호사/회계사...)", "전문직"),
    ("중.고등학교", "중고등학교"),
    ("중·고등학교", "중고등학교"),
    ("중ㆍ고등학교", "중고등학교"),
    ("중/고등학교", "중고등학교"),
    ("중,고등학교", "중고등학교"),
    ("복지관/복지관련 기관", "복지관/복지관련기관"),
    ("장애인관련기관/센터", "장애인관련기관센터"),
    ("문화부체육관광부", "문화체육관광부"),
    ("첨단기술/AI/빅데이터 관련", "첨단기술/IT/빅데이터 관련"),
    ("산업통상자원부/특허청", "산업통산자원부/특허청"),
    ("리서치/텔레마케팅 회사", "리서치/텔레마케팅"),
    ("미술학원/애니메이션", "마술학원/애니메이션"),
    ("사무기기/컴퓨터 판매/정수기렌탈", "사무기기/컴퓨터 판매/렌탈"),
    ("국제교류/협력관련", "국제교류관련"),
    ("귀금속/악세사리", "귀금속/악세서리"),
    ("보건복지부/질병관리청", "보건복지부/질병관리본부"),
    ("신용카드/캐피탈", "산용카드/캐피탈"),
    ("장애인종합복지관", "장애인복지관"),
    ("국민건강보험공단/평가원", "국민건강보험공단"),
    ("에너지/배터리/태양열/정유", "에너지/태양열/정유"),
    ("애견용품/동물병원/반려관련", "애견용품/동물병원/애견관련"),
    ("영상의학과의원", "영상의학과"),
    ("청와대/국회/정당", "청와대/국회"),
    ("PC방/보드카페/게임방", "PC방/보드카페"),
    ("대학교박물관/기념관", "대학교박물관"),
    ("입학/홍보/기획", "입학/홍보"),
    ("학생처/학생지원센터", "학생지원센터"),
    ("청년회의소(JCI)", "청년회의소"),
    ("교수학습지원센터/교수협의회", "교수학습지원센터"),
    ("보안/경비/건물관리/용역", "보안경비/용역"),
    ("인터넷쇼핑몰/홈쇼핑", "인터넷쇼핑몰/앱"),
    ("학생상담센터/인권센터", "학생상담센터"),
    ("행정복지센터(주민센터)", "행정복지센터"),
    ("학습지/방문교육", "학습지/학습교육"),
    ("한국EMS협회", "한국EMS연맹"),
    ("엔터테인먼트/연예기획사", "엔터테인먼트"),
    ("소방청/소방서", "소방서"),
    ("기타 교육관련기관", "기타교육관련기관"),
    ("근로자의날 기념", "근로자의 날 기념"),
    ("어버이날/스승의날기념", "어버이날/스승의날 기념"),
    ("가족센터(건강가정.다문화가족)", "가족센터(건강가정,다문화가족)"),
    ("여성관련 협회/재단", "여성관련협회/재단"),
    ("여성긴급전화 1366", "여성긴급전화1366"),
    ("여성인력개발센터 (여성새로일하기센터)", "여성인력개발센터(여성새로일하기센터)"),
    ("칠순.팔순.구순기념", "칠순팔순구순기념"),
    ("기타 어린이관련학교", "기타어린이관련학교"),
]

_alias_map = {}
for a, b in CATEGORY_ALIAS_PAIRS:
    na = normalize_category_raw(a)
    nb = normalize_category_raw(b)
    canonical = min(na, nb)
    _alias_map[na] = canonical
    _alias_map[nb] = canonical


def normalize_category(value):
    raw = normalize_category_raw(value)
    return _alias_map.get(raw, raw)


def parse_one_date(value):
    """엑셀 날짜 숫자, datetime, YYYY/MM/DD, YYYY-MM-DD 혼합 처리."""
    if pd.isna(value):
        return ""

    if isinstance(value, pd.Timestamp):
        return value.strftime("%Y-%m-%d")
    if isinstance(value, datetime):
        return value.strftime("%Y-%m-%d")
    if isinstance(value, date):
        return value.strftime("%Y-%m-%d")

    text = clean_text(value)
    if not text:
        return ""

    text_num = text.replace(",", "")
    if re.fullmatch(r"\d+(\.0)?", text_num):
        try:
            num = float(text_num)
            if 20000 <= num <= 60000:
                return pd.to_datetime(num, unit="D", origin="1899-12-30").strftime("%Y-%m-%d")
        except Exception:
            pass

    text2 = text.replace(".", "-").replace("/", "-")
    try:
        dt = pd.to_datetime(text2, errors="coerce", format="mixed")
    except TypeError:
        dt = pd.to_datetime(text2, errors="coerce")

    if pd.isna(dt):
        return ""
    return dt.strftime("%Y-%m-%d")


def find_col(df, candidates, required=True):
    """컬럼명을 공백 제거 기준으로 찾아줌."""
    col_map = {clean_text(c): c for c in df.columns}
    for cand in candidates:
        key = clean_text(cand)
        if key in col_map:
            return col_map[key]
    if required:
        raise KeyError(f"컬럼을 찾지 못했습니다: {candidates}")
    return None


def find_best_col_by_nonblank(df, candidates, exclude_values=("-",)):
    """동일한 이름 후보가 여러 개일 때, 유효값이 가장 많은 컬럼을 선택."""
    clean_candidates = [clean_text(x) for x in candidates]
    matches = []
    for col in df.columns:
        if clean_text(col) in clean_candidates:
            ser = df[col].map(clean_text)
            valid = ser.ne("") & ~ser.isin(exclude_values)
            matches.append((int(valid.sum()), col))
    if not matches:
        raise KeyError(f"컬럼을 찾지 못했습니다: {candidates}")
    matches.sort(key=lambda x: x[0], reverse=True)
    return matches[0][1], matches


## 3. 컬럼 자동 탐색


In [ ]:
# 판촉사랑 컬럼
COL_87_KEY = find_col(df87, ["수집키"], required=False)
COL_87_CASE_ID = find_col(df87, ["납품사례ID"], required=False)
COL_87_PRODUCT_CODE = find_col(df87, ["상품코드"], required=False)
COL_87_PRODUCT = find_col(df87, ["상품명"])
COL_87_PRINT = find_col(df87, ["인쇄방법"], required=False)
COL_87_DATE = find_col(df87, ["등록일", "날짜"])
COL_87_LARGE = find_col(df87, ["구매처분류(대)", "구매처 분류(대)"], required=False)
COL_87_MID = find_col(df87, ["구매처분류(중)", "구매처 분류(중)"])
COL_87_SMALL = find_col(df87, ["구매처분류(소)", "구매처 분류(소)"], required=False)
COL_87_URL = find_col(df87, ["상세URL"], required=False)

# 거래데이터 컬럼
COL_TR_NO = find_col(dftr, ["번호"])
COL_TR_LARGE = find_col(dftr, ["구매처 분류(대)", "구매처분류(대)"])
COL_TR_MID = find_col(dftr, ["구매처 분류(중)", "구매처분류(중)"])
COL_TR_SMALL = find_col(dftr, ["구매처 분류(소)", "구매처분류(소)"], required=False)
COL_TR_DETAIL = find_col(dftr, ["구매처 분류(세)", "구매처분류(세)"], required=False)
COL_TR_PRODUCT = find_col(dftr, ["상품", "상품명"])
COL_TR_DATE = find_col(dftr, ["날짜", "등록일"])
COL_TR_PRINT = find_col(dftr, ["인쇄 방법", "인쇄방법"], required=False)
COL_TR_BUYER, buyer_col_matches = find_best_col_by_nonblank(
    dftr,
    ["구매처 명 ", "구매처 명", "구매처명"],
    exclude_values=("-",)
)

log(f"[{SOURCE_LABEL}]")
for name, col in [
    ("수집키", COL_87_KEY),
    ("납품사례ID", COL_87_CASE_ID),
    ("상품코드", COL_87_PRODUCT_CODE),
    ("상품명", COL_87_PRODUCT),
    ("인쇄방법", COL_87_PRINT),
    ("등록일", COL_87_DATE),
    ("구매처분류(대)", COL_87_LARGE),
    ("구매처분류(중)", COL_87_MID),
    ("구매처분류(소)", COL_87_SMALL),
    ("상세URL", COL_87_URL),
]:
    log(f"- {name}: {col}")


# 컬럼 위치 검수: 위치가 달라도 아래처럼 컬럼명으로 찾아서 사용합니다.
# 현재 업로드 파일 기준 예시: 상품코드=B열, 상품명=C열, 구매처분류(중)=M열, 구매처분류(소)=N열입니다.
log("\n[판촉사랑 컬럼 자동 탐색 완료]")
log("- 상품코드 컬럼:", COL_87_PRODUCT_CODE)
log("- 상품명 컬럼:", COL_87_PRODUCT)
log("- 구매처분류(중) 컬럼:", COL_87_MID)
log("- 구매처분류(소) 컬럼:", COL_87_SMALL)
if COL_87_PRODUCT_CODE:
    log("- 상품코드 앞자리 0 포함 예시:")
    display(df87[[COL_87_PRODUCT_CODE, COL_87_PRODUCT]].head(10))

log("\n[거래데이터]")
for name, col in [
    ("번호", COL_TR_NO),
    ("구매처 분류(대)", COL_TR_LARGE),
    ("구매처 분류(중)", COL_TR_MID),
    ("구매처 분류(소)", COL_TR_SMALL),
    ("구매처 분류(세)", COL_TR_DETAIL),
    ("구매처 명", COL_TR_BUYER),
    ("날짜", COL_TR_DATE),
    ("상품", COL_TR_PRODUCT),
    ("인쇄 방법", COL_TR_PRINT),
]:
    log(f"- {name}: {col}")

log("\n구매처명 후보 컬럼별 유효값 수:", buyer_col_matches)


## 4. 키 생성


In [ ]:
df87_work = df87.copy().astype("object")
dftr_work = dftr.copy().astype("object")

# 원본 행 정보: 엑셀 데이터 행은 header 다음부터 시작하므로 index + 2
df87_work["_87_order"] = range(len(df87_work))
df87_work["_87_excel_row"] = df87_work.index + 2
dftr_work["_trade_order"] = range(len(dftr_work))
dftr_work["_trade_excel_row"] = dftr_work.index + 2

# 날짜/상품명 키
df87_work["_key_date"] = df87_work[COL_87_DATE].map(parse_one_date)
dftr_work["_key_date"] = dftr_work[COL_TR_DATE].map(parse_one_date)

df87_work["_key_product_raw"] = df87_work[COL_87_PRODUCT].map(normalize_product_raw)
dftr_work["_key_product_raw"] = dftr_work[COL_TR_PRODUCT].map(normalize_product_raw)

df87_work["_key_product_exact"] = df87_work[COL_87_PRODUCT].map(normalize_product_exact)
dftr_work["_key_product_exact"] = dftr_work[COL_TR_PRODUCT].map(normalize_product_exact)

# relaxed 보정정규화 키
# 상품명 exact가 실패했을 때, 상태/이벤트 문구·사이즈·1P·우산 살대 수 등
# 상품 식별에 직접적이지 않은 반복 표현을 보정한 뒤 비교합니다.
df87_work["_key_product_relaxed"] = df87_work[COL_87_PRODUCT].map(normalize_product_relaxed)
dftr_work["_key_product_relaxed"] = dftr_work[COL_TR_PRODUCT].map(normalize_product_relaxed)

# 인쇄방법 키
df87_work["_key_print"] = df87_work[COL_87_PRINT].map(normalize_category) if COL_87_PRINT else ""
dftr_work["_key_print"] = dftr_work[COL_TR_PRINT].map(normalize_category) if COL_TR_PRINT else ""

# 분류 경로 키용 원본 경로 저장
def get_87_path(row):
    items = []
    if USE_87_LARGE_IN_CATEGORY_PATH and COL_87_LARGE:
        items.append(row[COL_87_LARGE])
    items.append(row[COL_87_MID])
    if COL_87_SMALL:
        items.append(row[COL_87_SMALL])
    return items


def get_trade_path(row):
    items = [row[COL_TR_LARGE], row[COL_TR_MID]]
    if COL_TR_SMALL:
        items.append(row[COL_TR_SMALL])
    if COL_TR_DETAIL:
        items.append(row[COL_TR_DETAIL])
    return items


df87_work["_category_path"] = df87_work.apply(get_87_path, axis=1)
dftr_work["_category_path"] = dftr_work.apply(get_trade_path, axis=1)

# 거래데이터 번호 숫자화. 정렬 보조용이며, 1대1 검증은 index 기준으로 함.
dftr_work["_trade_no_num"] = pd.to_numeric(dftr_work[COL_TR_NO].map(clean_text), errors="coerce")

# 날짜 기준 거래데이터 그룹.
# 날짜는 무조건 일치해야 하므로, 매칭과 미매칭 진단 모두 같은 날짜 그룹 안에서만 수행합니다.
trade_groups_by_date = {}

for idx, row in dftr_work.iterrows():
    date_key = row["_key_date"]
    if date_key:
        trade_groups_by_date.setdefault(date_key, []).append(idx)

log(f"{SOURCE_LABEL} 날짜 빈값:", int((df87_work["_key_date"] == "").sum()))
log("거래데이터 날짜 빈값:", int((dftr_work["_key_date"] == "").sum()))
log("거래데이터 날짜 그룹 수:", len(trade_groups_by_date))
log(f"{SOURCE_LABEL} 상품명 exact 빈값:", int((df87_work["_key_product_exact"] == "").sum()))
log("거래데이터 상품명 exact 빈값:", int((dftr_work["_key_product_exact"] == "").sum()))
log(f"{SOURCE_LABEL} 상품명 exact 정규화 변경 행 수:", int((df87_work["_key_product_raw"] != df87_work["_key_product_exact"]).sum()))
log("거래데이터 상품명 exact 정규화 변경 행 수:", int((dftr_work["_key_product_raw"] != dftr_work["_key_product_exact"]).sum()))
log(f"{SOURCE_LABEL} 상품명 relaxed 보정 변경 행 수:", int((df87_work["_key_product_exact"] != df87_work["_key_product_relaxed"]).sum()))
log("거래데이터 상품명 relaxed 보정 변경 행 수:", int((dftr_work["_key_product_exact"] != dftr_work["_key_product_relaxed"]).sum()))
log(f"{SOURCE_LABEL} 분류경로 예시:")
display(df87_work[[COL_87_MID] + ([COL_87_SMALL] if COL_87_SMALL else []) + ["_category_path"]].head(10))


## 5-1. 배정 순서 · 분류 예외 · 거래데이터 인덱스 준비


In [ ]:
# 거래데이터 후보 정렬 함수
if TRADE_ORDER_MODE == "번호순서":
    def sort_trade_indexes(indexes):
        return sorted(
            list(indexes),
            key=lambda i: (
                pd.isna(dftr_work.at[i, "_trade_no_num"]),
                dftr_work.at[i, "_trade_no_num"] if not pd.isna(dftr_work.at[i, "_trade_no_num"]) else 10**18,
                dftr_work.at[i, "_trade_order"],
            )
        )
else:
    def sort_trade_indexes(indexes):
        return sorted(list(indexes), key=lambda i: dftr_work.at[i, "_trade_order"])


# ------------------------------------------------------------
# 빠른 단계별 필터용 키 생성
# ------------------------------------------------------------
# 사람의 엑셀 필터 방식과 동일하게 처리합니다.
# 공통 조건:
# 1) 거래데이터 구매처 분류(중) 컬럼에서 찾기
# 2) 중분류가 맞은 거래데이터 안에서 구매처 분류(소) 컬럼 찾기
# 3) 중+소분류가 맞은 거래데이터 안에서 판촉사랑 등록일만 확인
#
# 상품명 비교 순서:
# 4-1) 상품명 완전일치(raw)
# 4-2) 상품명 exact 기본정규화
# 4-3) 상품명 relaxed 보정정규화
# 4-4) BERT 상품명 의미 유사도
#
# 전부 맞으면 미사용 거래데이터를 거래데이터 행 순서대로 1대1 배정합니다.


def norm_mid(value):
    return normalize_category(value)


def norm_small(value):
    return normalize_category(value)


# 87 / 거래데이터의 중분류, 소분류 비교 키
# 원본 컬럼은 건드리지 않고 비교용 키만 추가합니다.
df87_work["_key_mid"] = df87_work[COL_87_MID].map(norm_mid)
df87_work["_key_small"] = df87_work[COL_87_SMALL].map(norm_small) if COL_87_SMALL else ""

dftr_work["_key_mid"] = dftr_work[COL_TR_MID].map(norm_mid)
dftr_work["_key_small"] = dftr_work[COL_TR_SMALL].map(norm_small) if COL_TR_SMALL else ""


# ------------------------------------------------------------
# 분류 예외 매핑 - 7번 수정
# ------------------------------------------------------------
# 원본 분류값은 절대 수정하지 않고, 매칭용 후보 키에만 예외를 적용합니다.
#
# 거래데이터:
#   구매처분류(중) = 전시회/박람회/축제/행사
#
# 판촉사랑:
#   구매처분류(중) = 기념행사별 또는 전시회
#
# 위 구조에서는 판촉사랑의 두 중분류 아래 구매처분류(소)를 합쳐서
# 거래데이터의 구매처분류(소)와 비교해야 하므로 1:N 중분류 예외로 처리합니다.
TRADE_TO_PROMO_MID_EXCEPTIONS = {
    norm_mid("전시회/박람회/축제/행사"): [norm_mid("기념행사별"), norm_mid("전시회")],
}

PROMO_TO_TRADE_MID_EXCEPTIONS = {}
for _trade_mid, _promo_mid_list in TRADE_TO_PROMO_MID_EXCEPTIONS.items():
    for _promo_mid in _promo_mid_list:
        PROMO_TO_TRADE_MID_EXCEPTIONS.setdefault(_promo_mid, []).append(_trade_mid)


def unique_keep_order(values):
    """순서를 유지하면서 중복 제거. 문자열은 공백 정리, 숫자 index는 원형 유지."""
    out = []
    seen = set()
    for value in values:
        if pd.isna(value):
            continue
        key = clean_text(value) if isinstance(value, str) else value
        if key == "" or key in seen:
            continue
        seen.add(key)
        out.append(key)
    return out


def collect_sorted_indexes(indexes):
    """여러 후보 index를 중복 제거 후 거래데이터 기준 순서로 정렬."""
    return sort_trade_indexes(unique_keep_order(indexes))


def get_trade_mid_candidates_for_promo(promo_mid):
    """판촉사랑 중분류 1개가 거래데이터에서 확인해야 할 중분류 후보 목록."""
    promo_mid = clean_text(promo_mid)
    return unique_keep_order([promo_mid] + PROMO_TO_TRADE_MID_EXCEPTIONS.get(promo_mid, []))


# ------------------------------------------------------------
# 분류 예외 추가: 관광지/국립공원/놀이공원 ↔ 골프관련
# ------------------------------------------------------------
# 원본 분류값은 절대 수정하지 않고, 매칭용 후보 키에만 예외를 적용합니다.
#
# 거래데이터:
#   구매처분류(중) = 골프관련
#   구매처분류(소) = 관광지/국립공원/놀이공원
#
# 판촉사랑:
#   구매처분류(중) = 관광지/국립공원/놀이공원
#   구매처분류(소) = 관광지/유적지, 국립공원/수목원, 놀이공원, 기타
#
# 위 구조에서는 판촉사랑의 관광지 하위 소분류를
# 거래데이터의 골프관련 / 관광지/국립공원/놀이공원 후보와도 비교합니다.
# 날짜 조건과 상품명 매칭 순서는 기존 로직 그대로 유지합니다.
TOUR_PROMO_MID = norm_mid("관광지/국립공원/놀이공원")
TOUR_TRADE_MID = norm_mid("골프관련")
TOUR_TRADE_SMALL = norm_small("관광지/국립공원/놀이공원")

TOUR_PROMO_SMALLS = {
    norm_small("관광지/유적지"),
    norm_small("국립공원/수목원"),
    norm_small("놀이공원"),
    norm_small("기타"),
}


def get_trade_filter_pairs_for_promo(promo_mid, promo_small):
    """판촉사랑 행 기준으로 거래데이터에서 확인할 (중분류, 소분류) 후보 목록."""
    promo_mid = clean_text(promo_mid)
    promo_small = clean_text(promo_small)

    pairs = [(mid, promo_small) for mid in get_trade_mid_candidates_for_promo(promo_mid)]

    # 관광지/골프관련 예외 추가
    # 판촉사랑: 관광지/국립공원/놀이공원 / 관광지/유적지, 국립공원/수목원, 놀이공원, 기타
    # 거래데이터: 골프관련 / 관광지/국립공원/놀이공원
    if promo_mid == TOUR_PROMO_MID and promo_small in TOUR_PROMO_SMALLS:
        pairs.append((TOUR_TRADE_MID, TOUR_TRADE_SMALL))

    return unique_keep_order(pairs)


def get_promo_mid_candidates_for_trade(trade_mid):
    """미사용 거래데이터 진단용: 거래데이터 중분류가 판촉사랑에서 확인해야 할 중분류 후보 목록."""
    trade_mid = clean_text(trade_mid)
    return unique_keep_order([trade_mid] + TRADE_TO_PROMO_MID_EXCEPTIONS.get(trade_mid, []))


def get_promo_filter_pairs_for_trade(trade_mid, trade_small):
    """미사용 거래데이터 진단용: 거래데이터 행 기준으로 판촉사랑에서 확인할 (중분류, 소분류) 후보 목록.

    get_trade_filter_pairs_for_promo의 역방향입니다. 관광지/골프관련 예외도 반대 방향으로
    동일하게 반영해야, 정방향에서는 매칭 가능한 조합인데 미사용 진단에서만
    "매칭안됨"으로 잘못 표시되는 것을 막을 수 있습니다.
    """
    trade_mid = clean_text(trade_mid)
    trade_small = clean_text(trade_small)

    pairs = [(mid, trade_small) for mid in get_promo_mid_candidates_for_trade(trade_mid)]

    # 관광지/골프관련 예외 역방향
    if trade_mid == TOUR_TRADE_MID and trade_small == TOUR_TRADE_SMALL:
        pairs.extend((TOUR_PROMO_MID, small) for small in TOUR_PROMO_SMALLS)

    return unique_keep_order(pairs)


def collect_from_mapping(mapping, keys):
    """여러 key에 해당하는 후보 index를 합친 뒤 정렬."""
    out = []
    for key in keys:
        out.extend(mapping.get(key, []))
    return collect_sorted_indexes(out)


# ------------------------------------------------------------
# 거래데이터를 단계별 필터 키로 미리 그룹화합니다.
# ------------------------------------------------------------
# 날짜 목록 전체를 만들지 않습니다.
# 필요한 것은 "해당 날짜가 있는지"와 "해당 날짜 후보 index"뿐입니다.

trade_indexes_sorted = sort_trade_indexes(dftr_work.index.tolist())

trade_by_mid = {}
trade_by_mid_small = {}
trade_by_mid_small_date = {}
trade_by_mid_small_date_raw = {}
trade_by_mid_small_date_exact = {}
trade_by_mid_small_date_relaxed = {}

for idxtr in trade_indexes_sorted:
    mid = clean_text(dftr_work.at[idxtr, "_key_mid"])
    small = clean_text(dftr_work.at[idxtr, "_key_small"])
    date_key = clean_text(dftr_work.at[idxtr, "_key_date"])
    p_raw = clean_text(dftr_work.at[idxtr, "_key_product_raw"])
    p_exact = clean_text(dftr_work.at[idxtr, "_key_product_exact"])
    p_relaxed = clean_text(dftr_work.at[idxtr, "_key_product_relaxed"])

    if not mid:
        continue

    # 1단계: 중분류 필터
    trade_by_mid.setdefault(mid, []).append(idxtr)

    # 2단계: 중분류 + 소분류 필터
    ms_key = (mid, small)
    trade_by_mid_small.setdefault(ms_key, []).append(idxtr)

    # 3단계: 중분류 + 소분류 + 해당 날짜 필터
    if date_key:
        msd_key = (mid, small, date_key)
        trade_by_mid_small_date.setdefault(msd_key, []).append(idxtr)

        # 4단계: 상품명 필터용 색인
        if p_raw:
            trade_by_mid_small_date_raw.setdefault((mid, small, date_key, p_raw), []).append(idxtr)

        if p_exact:
            trade_by_mid_small_date_exact.setdefault((mid, small, date_key, p_exact), []).append(idxtr)

        if p_relaxed:
            trade_by_mid_small_date_relaxed.setdefault((mid, small, date_key, p_relaxed), []).append(idxtr)


## 5-2. BERT 유사도 함수 정의


In [ ]:
# ------------------------------------------------------------
# BERT 상품명 의미 유사도 함수 - GPU 사용
# ------------------------------------------------------------
# check_cuda_or_raise는 위쪽 "0. BERT 사용 환경 사전 점검" 단계로 옮겼습니다.
# 여기서는 이미 정의된 그 함수를 그대로 사용합니다.
bert_model = None


def load_embedding_cache():
    """디스크에 저장된 임베딩 캐시를 불러옵니다. {모델명: {상품명: 임베딩}} 구조라 모델을 바꿔도 안 섞입니다."""
    if EMBEDDING_CACHE_FILE.exists():
        try:
            with open(EMBEDDING_CACHE_FILE, "rb") as f:
                cache = pickle.load(f)
            log(
                "임베딩 캐시 불러옴:", EMBEDDING_CACHE_FILE,
                f"(모델 {BERT_MODEL_NAME} 캐시 {len(cache.get(BERT_MODEL_NAME, {}))}건)",
            )
            return cache
        except Exception as exc:
            log("임베딩 캐시를 불러오지 못해 새로 시작합니다:", exc)
    return {}


def save_embedding_cache():
    """임베딩 캐시를 디스크에 저장합니다. 중간에 멈춰도 이미 계산한 임베딩은 다시 계산하지 않도록 합니다."""
    EMBEDDING_CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
    with open(EMBEDDING_CACHE_FILE, "wb") as f:
        pickle.dump(_embedding_cache, f)


_embedding_cache = load_embedding_cache()


def get_bert_model():
    """SentenceTransformer 모델을 최초 1회만 로드합니다."""
    global bert_model
    if bert_model is None:
        try:
            from sentence_transformers import SentenceTransformer
        except ImportError as exc:
            raise ImportError(
                "BERT 상품명 유사도를 사용하려면 sentence-transformers 설치가 필요합니다. "
                "노트북에서 `pip install sentence-transformers` 실행 후 다시 실행하세요."
            ) from exc

        device = check_cuda_or_raise() if BERT_DEVICE == "cuda" else BERT_DEVICE
        log("BERT 모델 로딩 중:", BERT_MODEL_NAME)
        log("BERT device:", device)
        bert_model = SentenceTransformer(BERT_MODEL_NAME, device=device)
        log("BERT 모델 로딩 완료")
    return bert_model


def get_product_embedding(text):
    """상품명 1개를 embedding으로 변환하고 cache로 재사용합니다(모델별로 디스크에도 저장)."""
    text = clean_text(text)
    if not text:
        return None

    model_cache = _embedding_cache.setdefault(BERT_MODEL_NAME, {})
    if text in model_cache:
        return model_cache[text]

    model = get_bert_model()
    emb = model.encode(
        text,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )
    model_cache[text] = emb
    return emb


def cosine_sim(a, b):
    if a is None or b is None:
        return -1.0
    return float(a @ b)


def find_bert_product_candidates(query_product, candidate_indexes, threshold=None):
    """같은 중분류+소분류+날짜 후보 안에서만 BERT 유사도 비교.

    반환값:
    - bert_indexes: threshold 이상 후보 index 목록
    - bert_scores: {거래데이터 index: 유사도 점수}
    """
    threshold = BERT_SIM_THRESHOLD if threshold is None else threshold
    query_product = clean_text(query_product)
    candidate_indexes = sort_trade_indexes(list(dict.fromkeys(candidate_indexes)))

    if not USE_BERT_PRODUCT_SIMILARITY:
        return [], {}

    if not query_product or not candidate_indexes:
        return [], {}

    # BERT 후보 수 제한 없음:
    # 이미 중분류+소분류+날짜 조건으로 후보를 줄였으므로,
    # 해당 날짜 후보 전체를 대상으로 상품명 유사도를 계산합니다.

    q_emb = get_product_embedding(query_product)
    scored = []
    score_map = {}

    for idxtr in candidate_indexes:
        cand_product = clean_text(dftr_work.at[idxtr, "_key_product_relaxed"]) or clean_text(dftr_work.at[idxtr, "_key_product_exact"]) or clean_text(dftr_work.at[idxtr, "_key_product_raw"])
        if not cand_product:
            continue

        c_emb = get_product_embedding(cand_product)
        score = cosine_sim(q_emb, c_emb)
        score_map[idxtr] = score

        if score >= threshold:
            scored.append((idxtr, score))

    # 유사도 높은 순서, 같으면 거래데이터 원본 순서
    scored.sort(key=lambda x: (-x[1], dftr_work.at[x[0], "_trade_order"]))

    return [idx for idx, score in scored], score_map


# ------------------------------------------------------------
# 진단용 문자열 함수
# ------------------------------------------------------------
def limited_join(values, limit=30):
    """진단용 문자열이 너무 길어지지 않도록 제한해서 표시합니다."""
    vals = [clean_text(v) for v in values if clean_text(v)]
    seen = []
    for v in vals:
        if v not in seen:
            seen.append(v)
    if len(seen) > limit:
        return " / ".join(seen[:limit]) + f" / ...외 {len(seen) - limit}개"
    return " / ".join(seen)


def join_trade_no(indexes, limit=30):
    """거래데이터 번호를 / 로 묶어 표시합니다."""
    indexes = sort_trade_indexes(list(dict.fromkeys(indexes)))
    if not indexes:
        return ""
    return limited_join(dftr_work.loc[indexes, COL_TR_NO].map(clean_text).tolist(), limit=limit)


def get_bert_score_text(indexes, score_map, limit=30):
    """후보별 BERT 점수를 사람이 보기 쉽게 표시합니다."""
    items = []
    for idxtr in indexes[:limit]:
        no = clean_text(dftr_work.at[idxtr, COL_TR_NO])
        score = score_map.get(idxtr, "")
        if score == "":
            items.append(no)
        else:
            items.append(f"{no}:{score:.4f}")
    if len(indexes) > limit:
        items.append(f"...외 {len(indexes) - limit}개")
    return " / ".join(items)


## 5-3. 매칭 함수 정의 및 결과 컬럼 준비


In [ ]:
# ------------------------------------------------------------
# 판촉사랑 1행 기준 후보 추출
# ------------------------------------------------------------
def get_trade_candidates_by_stage(idx87, enable_bert=True):
    """
    판촉사랑 1행에 대해 사람의 필터 방식으로 단계별 후보를 반환합니다.

    상태값 판단 기준:
    - 중분류 후보가 없으면: 미매칭_구매처분류(중)_매칭안됨
    - 중분류는 있으나 소분류 후보가 없으면: 미매칭_구매처분류(소)_매칭안됨
    - 중/소분류는 있으나 해당 날짜 후보가 없으면: 미매칭_날짜없음
    - 중/소분류+날짜는 있으나 상품명 후보가 없으면: 미매칭_상품명매칭안됨
    - 전부 맞으면: 1대1 매칭
    """
    row87 = df87_work.loc[idx87]

    mid = clean_text(row87["_key_mid"])
    small = clean_text(row87["_key_small"])
    date_key = clean_text(row87["_key_date"])
    p_raw = clean_text(row87["_key_product_raw"])
    p_exact = clean_text(row87["_key_product_exact"])
    p_relaxed = clean_text(row87["_key_product_relaxed"])

    # 7번 수정: 특정 거래데이터 중분류가 판촉사랑에서 여러 중분류로 나뉜 경우를 처리합니다.
    # 예: 거래데이터 중분류 "전시회/박람회/축제/행사"는
    #     판촉사랑 중분류 "기념행사별", "전시회" 후보와 호환 처리합니다.
    filter_pairs = get_trade_filter_pairs_for_promo(mid, small) if mid else []
    mid_candidates = unique_keep_order([pair[0] for pair in filter_pairs])

    mid_indexes = collect_from_mapping(trade_by_mid, mid_candidates) if mid_candidates else []
    mid_small_indexes = collect_from_mapping(trade_by_mid_small, filter_pairs) if filter_pairs else []
    mid_small_date_keys = [(m, s, date_key) for m, s in filter_pairs] if (filter_pairs and date_key) else []
    mid_small_date_indexes = collect_from_mapping(trade_by_mid_small_date, mid_small_date_keys) if mid_small_date_keys else []

    # 1순위: 상품명 완전일치(raw)
    raw_keys = [(m, s, date_key, p_raw) for m, s in filter_pairs] if (filter_pairs and date_key and p_raw) else []
    raw_indexes = collect_from_mapping(trade_by_mid_small_date_raw, raw_keys) if raw_keys else []

    # 2순위: 상품명 exact 기본정규화
    exact_indexes = []
    if not raw_indexes and p_exact:
        exact_keys = [(m, s, date_key, p_exact) for m, s in filter_pairs] if (filter_pairs and date_key) else []
        exact_indexes = collect_from_mapping(trade_by_mid_small_date_exact, exact_keys) if exact_keys else []

    # 3순위: 상품명 relaxed 보정정규화
    relaxed_indexes = []
    if not raw_indexes and not exact_indexes and p_relaxed:
        relaxed_keys = [(m, s, date_key, p_relaxed) for m, s in filter_pairs] if (filter_pairs and date_key) else []
        relaxed_indexes = collect_from_mapping(trade_by_mid_small_date_relaxed, relaxed_keys) if relaxed_keys else []

    # 4순위: BERT 상품명 의미 유사도
    bert_indexes = []
    bert_scores = {}
    if enable_bert and not raw_indexes and not exact_indexes and not relaxed_indexes and mid_small_date_indexes:
        # BERT는 상품명 완전일치/exact/relaxed가 모두 실패한 경우에만 실행합니다.
        # 비교 텍스트는 relaxed 보정키를 우선 사용하고, 없으면 exact/raw로 fallback합니다.
        bert_indexes, bert_scores = find_bert_product_candidates(
            query_product=p_relaxed or p_exact or p_raw,
            candidate_indexes=mid_small_date_indexes,
            threshold=BERT_SIM_THRESHOLD,
        )

    # 상품명 후보 전체: raw → exact → relaxed → BERT 순서
    product_indexes = sort_trade_indexes(set(raw_indexes) | set(exact_indexes) | set(relaxed_indexes) | set(bert_indexes))

    return {
        "mid": mid,
        "small": small,
        "date_key": date_key,
        "filter_pairs": filter_pairs,
        "mid_candidates": mid_candidates,
        "mid_indexes": mid_indexes,
        "mid_small_indexes": mid_small_indexes,
        "mid_small_date_indexes": mid_small_date_indexes,
        "raw_indexes": raw_indexes,
        "exact_indexes": exact_indexes,
        "relaxed_indexes": relaxed_indexes,
        "bert_indexes": bert_indexes,
        "bert_scores": bert_scores,
        "product_indexes": product_indexes,
        "mid_count": len(mid_indexes),
        "mid_small_count": len(mid_small_indexes),
        "mid_small_date_count": len(mid_small_date_indexes),
        "product_count": len(product_indexes),
    }


def choose_trade_index(stage_info, used_trade_indexes, allowed_modes=None):
    """
    실제 1대1 배정 후보를 선택합니다.

    핵심 변경점:
    - 전체 판촉사랑 행에 대해 raw → exact → relaxed 확정 매칭을 먼저 끝냅니다.
    - BERT는 마지막 단계에서만 실행합니다.
    - BERT 단계에서는 구매처분류(중)+구매처분류(소)+날짜 필터를 통과한 미사용 후보 중
      BERT 유사도 점수가 가장 높은 1개 거래데이터만 선택합니다.
    - 같은 점수일 때만 거래데이터 원본순서로 보조 정렬합니다.

    이렇게 해야 낮은 BERT 점수 후보가 뒤쪽의 상품명 완전일치/정규화 일치 후보를 먼저 빼앗지 않습니다.
    """
    if allowed_modes is None:
        allowed_modes = ("raw", "exact", "relaxed", "bert")
    allowed_modes = set(allowed_modes)

    raw_unused = [i for i in stage_info["raw_indexes"] if i not in used_trade_indexes]
    exact_unused = [i for i in stage_info["exact_indexes"] if i not in used_trade_indexes]
    relaxed_unused = [i for i in stage_info["relaxed_indexes"] if i not in used_trade_indexes]
    bert_unused = [i for i in stage_info["bert_indexes"] if i not in used_trade_indexes]

    if "raw" in allowed_modes and raw_unused:
        return sort_trade_indexes(raw_unused), "상품명완전일치", "raw"

    if "exact" in allowed_modes and exact_unused:
        return sort_trade_indexes(exact_unused), "상품명exact일치", "exact"

    if "relaxed" in allowed_modes and relaxed_unused:
        return sort_trade_indexes(relaxed_unused), "상품명relaxed일치", "relaxed"

    if "bert" in allowed_modes and bert_unused:
        score_map = stage_info.get("bert_scores", {})
        bert_unused = sorted(
            bert_unused,
            key=lambda i: (-score_map.get(i, 0), dftr_work.at[i, "_trade_order"])
        )
        # 실제 배정은 1건(passed_idxs[0])만 하되, 임계값을 넘는 나머지 후보도 함께
        # 반환해서 apply_success_match가 "후보다수_검토" 시트에 남기도록 합니다.
        return bert_unused, "BERT상품명유사도", "bert"

    return [], "", ""


# ------------------------------------------------------------
# 1대1 매칭 실행
# ------------------------------------------------------------
result_df = df87.copy().astype("object")

# 메인 시트에는 요청한 결과 컬럼을 추가/갱신합니다.
result_df["구매처분류(대)"] = ""
result_df["구매처명"] = ""
result_df["매칭상태"] = ""
result_df["매칭건수"] = 0
result_df["상품명매칭방식"] = ""
result_df["BERT유사도"] = pd.Series([""] * len(result_df), index=result_df.index, dtype="object")

# 미매칭 원인 진단용 컬럼
# 모든 후보 날짜/후보 번호를 전부 나열하지 않습니다.
# 사람의 필터 방식처럼 현재 확인하는 값과 해당 단계 후보 수만 남깁니다.
result_df["미매칭진단_확인단계"] = ""
result_df["미매칭진단_확인중분류"] = ""
result_df["미매칭진단_확인소분류"] = ""
result_df["미매칭진단_확인날짜"] = ""
result_df["미매칭진단_해당단계후보수"] = 0
result_df["미매칭진단_해당단계후보번호"] = pd.Series([""] * len(result_df), index=result_df.index, dtype="object")

# pandas 3.x 계열에서 빈 문자열 컬럼이 string dtype으로 고정되면,
# 나중에 BERT 점수(float)나 기타 값 입력 시 TypeError가 날 수 있으므로
# 결과 추가 컬럼은 명시적으로 object dtype으로 고정합니다.
for _col in [
    "구매처분류(대)", "구매처명", "매칭상태", "상품명매칭방식", "BERT유사도",
    "미매칭진단_확인단계", "미매칭진단_확인중분류", "미매칭진단_확인소분류",
    "미매칭진단_확인날짜", "미매칭진단_해당단계후보번호",
]:
    if _col in result_df.columns:
        result_df[_col] = result_df[_col].astype("object")

matched_detail_rows = []
unmatched_87_indexes = []
review_rows = []
used_trade_indexes = set()
matched_87_indexes = set()


def apply_success_match(idx87, stage_info, passed_idxs, product_reason, chosen_mode):
    """선택된 거래데이터 1건을 result_df와 상세시트에 반영합니다."""
    row87 = df87_work.loc[idx87]
    idxtr = passed_idxs[0]
    used_trade_indexes.add(idxtr)
    matched_87_indexes.add(idx87)

    result_df.at[idx87, "매칭건수"] = len(passed_idxs)
    result_df.at[idx87, "상품명매칭방식"] = product_reason

    buyer = clean_text(dftr_work.at[idxtr, COL_TR_BUYER])
    large = clean_text(dftr_work.at[idxtr, COL_TR_LARGE])

    bert_score = ""
    if chosen_mode == "bert":
        bert_score = stage_info.get("bert_scores", {}).get(idxtr, "")
        result_df.at[idx87, "BERT유사도"] = f"{float(bert_score):.4f}" if bert_score != "" else ""

    result_df.at[idx87, "구매처분류(대)"] = large
    if buyer not in EMPTY_BUYER_VALUES:
        result_df.at[idx87, "구매처명"] = buyer
        if chosen_mode == "raw":
            result_df.at[idx87, "매칭상태"] = "매칭_1대1_상품명완전일치"
        elif chosen_mode == "exact":
            result_df.at[idx87, "매칭상태"] = "매칭_1대1_상품명exact일치"
        elif chosen_mode == "relaxed":
            result_df.at[idx87, "매칭상태"] = "매칭_1대일_상품명relaxed일치".replace("1대일", "1대1")
        elif chosen_mode == "bert":
            result_df.at[idx87, "매칭상태"] = "매칭_1대1_BERT상품명유사도"
        else:
            result_df.at[idx87, "매칭상태"] = "매칭_1대1_기타"
    else:
        result_df.at[idx87, "구매처명"] = ""
        result_df.at[idx87, "매칭상태"] = "매칭_1대1_구매처명없음"

    # 후보가 2개 이상이면 검토 시트에 남김
    if len(passed_idxs) >= 2:
        review_rows.append({
            "판촉사랑_엑셀행": int(df87_work.at[idx87, "_87_excel_row"]),
            "판촉사랑_수집키": clean_text(df87_work.at[idx87, COL_87_KEY]) if COL_87_KEY else "",
            "판촉사랑_납품사례ID": clean_text(df87_work.at[idx87, COL_87_CASE_ID]) if COL_87_CASE_ID else "",
            "판촉사랑_상품명": clean_text(df87_work.at[idx87, COL_87_PRODUCT]),
            "판촉사랑_인쇄방법": clean_text(df87_work.at[idx87, COL_87_PRINT]) if COL_87_PRINT else "",
            "판촉사랑_구매처분류(중)": clean_text(df87_work.at[idx87, COL_87_MID]),
            "판촉사랑_구매처분류(소)": clean_text(df87_work.at[idx87, COL_87_SMALL]) if COL_87_SMALL else "",
            "판촉사랑_등록일": clean_text(df87_work.at[idx87, COL_87_DATE]),
            "상품명매칭방식": product_reason,
            "BERT유사도": round(float(bert_score), 4) if bert_score != "" else "",
            "분류통과후보수": len(passed_idxs),
            "후보_거래데이터번호": " / ".join(dftr_work.loc[passed_idxs, COL_TR_NO].map(clean_text).tolist()),
            "후보_구매처명": " / ".join(dftr_work.loc[passed_idxs, COL_TR_BUYER].map(clean_text).tolist()),
            "후보_BERT점수": get_bert_score_text(passed_idxs, stage_info.get("bert_scores", {})) if chosen_mode == "bert" else "",
            "선택_거래데이터번호": clean_text(dftr_work.at[idxtr, COL_TR_NO]),
            "선택_구매처명": buyer,
        })

    # 상세 시트용: 판촉사랑 식별값 + 거래데이터 원본 전체
    rec = {
        "판촉사랑_엑셀행": int(df87_work.at[idx87, "_87_excel_row"]),
        "판촉사랑_수집키": clean_text(df87_work.at[idx87, COL_87_KEY]) if COL_87_KEY else "",
        "판촉사랑_납품사례ID": clean_text(df87_work.at[idx87, COL_87_CASE_ID]) if COL_87_CASE_ID else "",
        "판촉사랑_상품코드": clean_text(df87_work.at[idx87, COL_87_PRODUCT_CODE]) if COL_87_PRODUCT_CODE else "",
        "판촉사랑_상품명": clean_text(df87_work.at[idx87, COL_87_PRODUCT]),
        "판촉사랑_상품명_raw키": row87["_key_product_raw"],
        "판촉사랑_상품명_exact키": row87["_key_product_exact"],
        "판촉사랑_상품명_relaxed키": row87["_key_product_relaxed"],
        "판촉사랑_인쇄방법": clean_text(df87_work.at[idx87, COL_87_PRINT]) if COL_87_PRINT else "",
        "판촉사랑_구매처분류(대)": clean_text(df87_work.at[idx87, COL_87_LARGE]) if (COL_87_LARGE and USE_87_LARGE_IN_CATEGORY_PATH) else "",
        "판촉사랑_구매처분류(중)": clean_text(df87_work.at[idx87, COL_87_MID]),
        "판촉사랑_구매처분류(소)": clean_text(df87_work.at[idx87, COL_87_SMALL]) if COL_87_SMALL else "",
        "판촉사랑_등록일": clean_text(df87_work.at[idx87, COL_87_DATE]),
        "판촉사랑_상세URL": clean_text(df87_work.at[idx87, COL_87_URL]) if COL_87_URL else "",
        "매칭_상품명방식": product_reason,
        "매칭_BERT유사도": round(float(bert_score), 4) if bert_score != "" else "",
        "매칭_분류방식": "구매처분류(중소)_예외포함" if stage_info.get("filter_pairs") and len(stage_info.get("mid_candidates", [])) >= 2 else "구매처분류(중소)_컬럼일치",
        "매칭_분류통과후보수": len(passed_idxs),
        "거래데이터_엑셀행": int(dftr_work.at[idxtr, "_trade_excel_row"]),
    }
    for col in dftr.columns:
        rec[col] = dftr_work.at[idxtr, col]
    matched_detail_rows.append(rec)


def apply_unmatched_status(idx87, stage_info):
    """최종 미매칭 행의 매칭상태를 단계별로 1개만 입력합니다."""
    row87 = df87_work.loc[idx87]

    result_df.at[idx87, "미매칭진단_확인중분류"] = clean_text(row87[COL_87_MID])
    result_df.at[idx87, "미매칭진단_확인소분류"] = clean_text(row87[COL_87_SMALL]) if COL_87_SMALL else ""
    result_df.at[idx87, "미매칭진단_확인날짜"] = clean_text(row87["_key_date"])
    result_df.at[idx87, "매칭건수"] = 0
    result_df.at[idx87, "상품명매칭방식"] = ""

    if stage_info["mid_count"] == 0:
        status = "미매칭_구매처분류(중)_매칭안됨"
        result_df.at[idx87, "미매칭진단_확인단계"] = "구매처분류(중)"
        result_df.at[idx87, "미매칭진단_해당단계후보수"] = 0
        result_df.at[idx87, "미매칭진단_해당단계후보번호"] = ""

    elif stage_info["mid_small_count"] == 0:
        status = "미매칭_구매처분류(소)_매칭안됨"
        result_df.at[idx87, "미매칭진단_확인단계"] = "구매처분류(소)"
        result_df.at[idx87, "미매칭진단_해당단계후보수"] = stage_info["mid_count"]
        result_df.at[idx87, "미매칭진단_해당단계후보번호"] = join_trade_no(stage_info["mid_indexes"])

    elif stage_info["mid_small_date_count"] == 0:
        status = "미매칭_날짜없음"
        result_df.at[idx87, "미매칭진단_확인단계"] = "날짜"
        result_df.at[idx87, "미매칭진단_해당단계후보수"] = 0
        result_df.at[idx87, "미매칭진단_해당단계후보번호"] = ""

    else:
        status = "미매칭_상품명매칭안됨"
        result_df.at[idx87, "미매칭진단_확인단계"] = "상품명"
        result_df.at[idx87, "미매칭진단_해당단계후보수"] = stage_info["mid_small_date_count"]
        result_df.at[idx87, "미매칭진단_해당단계후보번호"] = join_trade_no(stage_info["mid_small_date_indexes"])

    result_df.at[idx87, "매칭상태"] = status
    unmatched_87_indexes.append(idx87)


## 5-4. raw/exact/relaxed 매칭 실행


In [ ]:
# ------------------------------------------------------------
# 전체 우선순위 1~3단계: 정확/정규화 매칭을 전체 행에서 먼저 확정
# ------------------------------------------------------------
# 중요:
# 기존처럼 한 행씩 raw→exact→relaxed→BERT를 모두 처리하면,
# 낮은 BERT 점수 후보가 뒤쪽 행의 완전일치 거래데이터를 먼저 가져가는 문제가 생깁니다.
# 따라서 BERT는 모든 완전일치/exact/relaxed 배정이 끝난 뒤 마지막에만 실행합니다.

for mode in ["raw", "exact", "relaxed"]:
    for idx87 in df87_work.index:
        if idx87 in matched_87_indexes:
            continue

        stage_info = get_trade_candidates_by_stage(idx87, enable_bert=False)
        passed_idxs, product_reason, chosen_mode = choose_trade_index(
            stage_info,
            used_trade_indexes,
            allowed_modes=(mode,),
        )

        if passed_idxs:
            apply_success_match(idx87, stage_info, passed_idxs, product_reason, chosen_mode)


## 5-5. BERT 매칭 실행


In [ ]:
# 노트북에서 이 셀만 다시 실행해도, sentence-transformers/CUDA 문제를 5-4(raw/exact/relaxed)를
# 다시 돌리지 않고 바로 알 수 있도록 여기서 한 번 더 점검합니다(0번 사전 점검과 동일한 점검).
if USE_BERT_PRODUCT_SIMILARITY:
    try:
        import sentence_transformers  # noqa: F401
    except ImportError as exc:
        raise ImportError(
            "BERT 상품명 유사도를 사용하려면 sentence-transformers 설치가 필요합니다. "
            "노트북/터미널에서 `pip install sentence-transformers` 실행 후 다시 실행하세요."
        ) from exc

    if BERT_DEVICE == "cuda":
        check_cuda_or_raise()


# ------------------------------------------------------------
# 전체 우선순위 4단계: 남은 행만 BERT 최고 유사도 매칭
# ------------------------------------------------------------
# 임베딩 계산이 오래 걸리므로, 중간에 멈추더라도(오류/중단) 그때까지 계산한 임베딩은
# 잃어버리지 않도록 일정 건수마다 + 끝나면(성공/실패 상관없이) 캐시를 저장합니다.
_BERT_CACHE_SAVE_EVERY = 200

try:
    for _bert_i, idx87 in enumerate(df87_work.index):
        if idx87 in matched_87_indexes:
            continue

        stage_info = get_trade_candidates_by_stage(idx87, enable_bert=True)
        passed_idxs, product_reason, chosen_mode = choose_trade_index(
            stage_info,
            used_trade_indexes,
            allowed_modes=("bert",),
        )

        if passed_idxs:
            apply_success_match(idx87, stage_info, passed_idxs, product_reason, chosen_mode)

        if (_bert_i + 1) % _BERT_CACHE_SAVE_EVERY == 0:
            save_embedding_cache()
finally:
    save_embedding_cache()
    log(
        "임베딩 캐시 저장 완료:", EMBEDDING_CACHE_FILE,
        f"(모델 {BERT_MODEL_NAME} 캐시 {len(_embedding_cache.get(BERT_MODEL_NAME, {}))}건)",
    )


## 5-6. 최종 미매칭 상태 입력 · 미사용 거래데이터 진단 · 요약


In [ ]:
# ------------------------------------------------------------
# 최종 미매칭 상태 입력
# ------------------------------------------------------------
for idx87 in df87_work.index:
    if idx87 in matched_87_indexes:
        continue

    stage_info = get_trade_candidates_by_stage(idx87, enable_bert=True)
    apply_unmatched_status(idx87, stage_info)


matched_detail_df = pd.DataFrame(matched_detail_rows)
unmatched_87_df = result_df.loc[unmatched_87_indexes].copy() if unmatched_87_indexes else pd.DataFrame(columns=result_df.columns)


# ------------------------------------------------------------
# 미사용_거래데이터원본 시트용 매칭상태 1개 컬럼만 생성
# ------------------------------------------------------------
# 방향은 반대입니다.
# - 메인 매칭: 판촉사랑 행 -> 거래데이터 후보 찾기
# - 미사용 진단: 미사용 거래데이터 행 -> 판촉사랑 파일에 어느 단계까지 존재하는지 확인
#
# 표시 순서:
# 구매처분류(중) 없음 -> 구매처분류(소) 없음 -> 날짜없음 -> 상품명매칭안됨
# 조건이 모두 존재하지만 1대1 배정에서 선택되지 않은 거래데이터는 별도 상태 1개로만 표시합니다.

promo_by_mid = {}
promo_by_mid_small = {}
promo_by_mid_small_date = {}
promo_by_mid_small_date_raw = {}
promo_by_mid_small_date_exact = {}
promo_by_mid_small_date_relaxed = {}

for _idx87 in df87_work.index:
    _mid = norm_mid(df87_work.at[_idx87, COL_87_MID])
    _small = norm_small(df87_work.at[_idx87, COL_87_SMALL]) if COL_87_SMALL else ""
    _date = clean_text(df87_work.at[_idx87, "_key_date"])
    _raw = clean_text(df87_work.at[_idx87, "_key_product_raw"])
    _exact = clean_text(df87_work.at[_idx87, "_key_product_exact"])
    _relaxed = clean_text(df87_work.at[_idx87, "_key_product_relaxed"])

    if _mid:
        promo_by_mid.setdefault(_mid, []).append(_idx87)
    if _mid and _small:
        promo_by_mid_small.setdefault((_mid, _small), []).append(_idx87)
    if _mid and _small and _date:
        promo_by_mid_small_date.setdefault((_mid, _small, _date), []).append(_idx87)
    if _mid and _small and _date and _raw:
        promo_by_mid_small_date_raw.setdefault((_mid, _small, _date, _raw), []).append(_idx87)
    if _mid and _small and _date and _exact:
        promo_by_mid_small_date_exact.setdefault((_mid, _small, _date, _exact), []).append(_idx87)
    if _mid and _small and _date and _relaxed:
        promo_by_mid_small_date_relaxed.setdefault((_mid, _small, _date, _relaxed), []).append(_idx87)


def get_unused_trade_status(_idxtr):
    _mid = norm_mid(dftr_work.at[_idxtr, COL_TR_MID])
    _small = norm_small(dftr_work.at[_idxtr, COL_TR_SMALL]) if COL_TR_SMALL else ""
    _date = clean_text(dftr_work.at[_idxtr, "_key_date"])
    _raw = clean_text(dftr_work.at[_idxtr, "_key_product_raw"])
    _exact = clean_text(dftr_work.at[_idxtr, "_key_product_exact"])
    _relaxed = clean_text(dftr_work.at[_idxtr, "_key_product_relaxed"])

    # 7번 수정 + 관광지/골프관련 예외: 미사용 거래데이터 진단은 역방향이므로,
    # get_trade_filter_pairs_for_promo(정방향)가 허용하는 (중분류, 소분류) 조합을
    # get_promo_filter_pairs_for_trade로 반대 방향에서도 동일하게 확인합니다.
    _filter_pairs = get_promo_filter_pairs_for_trade(_mid, _small)
    _promo_mid_candidates = unique_keep_order([m for m, _ in _filter_pairs])
    _promo_ms_keys = _filter_pairs
    _promo_msd_keys = [(m, s, _date) for m, s in _filter_pairs]

    if not any(promo_by_mid.get(m) for m in _promo_mid_candidates):
        return "미사용_구매처분류(중)_매칭안됨"
    if not any(promo_by_mid_small.get(key) for key in _promo_ms_keys):
        return "미사용_구매처분류(소)_매칭안됨"
    if not any(promo_by_mid_small_date.get(key) for key in _promo_msd_keys):
        return "미사용_날짜없음"

    # 상품명은 메인 매칭과 동일하게 raw -> exact -> relaxed 순서로만 확인합니다.
    # 여기서는 미사용 거래데이터 시트에 상태 1개만 남기기 위해 BERT 재계산은 하지 않습니다.
    if _raw and any(promo_by_mid_small_date_raw.get((m, s, _date, _raw)) for m, s in _filter_pairs):
        return "미사용_조건일치_1대1미배정"
    if _exact and any(promo_by_mid_small_date_exact.get((m, s, _date, _exact)) for m, s in _filter_pairs):
        return "미사용_조건일치_1대1미배정"
    if _relaxed and any(promo_by_mid_small_date_relaxed.get((m, s, _date, _relaxed)) for m, s in _filter_pairs):
        return "미사용_조건일치_1대1미배정"

    return "미사용_상품명매칭안됨"


_unused_trade_indexes = [i for i in dftr_work.index if i not in used_trade_indexes]
unused_trade_df = dftr_work.loc[_unused_trade_indexes, dftr.columns].copy()
unused_trade_df.insert(0, "거래데이터_엑셀행", [int(i + 2) for i in unused_trade_df.index])
unused_trade_df.insert(1, "매칭상태", [get_unused_trade_status(i) for i in unused_trade_df.index])

review_df = pd.DataFrame(review_rows)

summary = pd.DataFrame([
    [f"{SOURCE_LABEL} 전체 행 수", len(df87)],
    ["거래데이터 전체 행 수", len(dftr)],
    ["1대1 매칭 완료 행 수", int(result_df["매칭상태"].astype(str).str.startswith("매칭_1대1").sum())],
    ["상품명완전일치 매칭 행 수", int((result_df["상품명매칭방식"] == "상품명완전일치").sum())],
    ["상품명exact 매칭 행 수", int((result_df["상품명매칭방식"] == "상품명exact일치").sum())],
    ["상품명relaxed 매칭 행 수", int((result_df["상품명매칭방식"] == "상품명relaxed일치").sum())],
    ["BERT 유사도 매칭 행 수", int((result_df["상품명매칭방식"] == "BERT상품명유사도").sum())],
    ["구매처명 입력 행 수", int(result_df["구매처명"].map(clean_text).ne("").sum())],
    ["매칭됐지만 구매처명 없음 행 수", int((result_df["매칭상태"] == "매칭_1대1_구매처명없음").sum())],
    [f"미매칭 {SOURCE_LABEL} 행 수", len(unmatched_87_df)],
    ["미매칭_구매처분류(중)_매칭안됨 행 수", int((result_df["매칭상태"] == "미매칭_구매처분류(중)_매칭안됨").sum())],
    ["미매칭_구매처분류(소)_매칭안됨 행 수", int((result_df["매칭상태"] == "미매칭_구매처분류(소)_매칭안됨").sum())],
    ["미매칭_날짜없음 행 수", int((result_df["매칭상태"] == "미매칭_날짜없음").sum())],
    ["미매칭_상품명매칭안됨 행 수", int((result_df["매칭상태"] == "미매칭_상품명매칭안됨").sum())],
    ["사용된 거래데이터 행 수", len(used_trade_indexes)],
    ["미사용 거래데이터 행 수", len(unused_trade_df)],
    ["후보다수 검토 행 수", len(review_df)],
    ["거래데이터 배정 순서", TRADE_ORDER_MODE],
    ["인쇄방법 조건", "제외 / 상세시트 확인용"],
    ["공통 필터 순서", "구매처분류(중) 컬럼 → 구매처분류(소) 컬럼 → 해당 날짜"],
    ["상품명 비교 순서", "상품명 완전일치 → 상품명 exact 기본정규화 → 상품명 relaxed 보정정규화 → BERT 유사도"],
    ["BERT 사용", USE_BERT_PRODUCT_SIMILARITY],
    ["BERT 모델", BERT_MODEL_NAME],
    ["BERT 자동매칭 기준", BERT_SIM_THRESHOLD],
    ["BERT 후보 수 제한", "없음"],
    ["사용하지 않는 상태값", "미매칭_1대1후보이미사용됨 / 미매칭_거래데이터수부족"],
    ["구매처명으로 선택된 거래데이터 컬럼", COL_TR_BUYER],
], columns=["항목", "값"])

log(summary.to_string(index=False))
log("\n매칭상태 분포:")
log(result_df["매칭상태"].value_counts(dropna=False).to_string())
log("\n상품명매칭방식 분포:")
log(result_df["상품명매칭방식"].replace("", "미매칭").value_counts(dropna=False).to_string())


## 6. 상품분류 보정 함수


In [ ]:
def clean_category_text(x):
    """상품분류 보정용 빈값 정리."""
    if pd.isna(x):
        return pd.NA
    x = str(x).strip()
    if x == "" or x.lower() == "nan":
        return pd.NA
    return x


def make_category_key(x):
    """
    상품분류 매칭용 key
    - 앞뒤 공백 제거
    - 중간 공백 제거
    """
    x = clean_category_text(x)
    if pd.isna(x):
        return pd.NA
    return re.sub(r"\s+", "", str(x))


def get_parenthesis_category_key(x):
    """
    예:
    볼펜(100원~500원미만) -> 100원~500원미만
    USB (스틱타입) -> 스틱타입
    보조배터리(무선충전 보조배터리)) -> 무선충전보조배터리
    """
    x = clean_category_text(x)
    if pd.isna(x):
        return pd.NA

    m = re.search(r"\(([^()]*)\)\)*\s*$", str(x))
    if m:
        return make_category_key(m.group(1))

    return pd.NA


def apply_product_category_fix(df, category_path):
    """
    category_reference.xlsx 기준으로 상품분류를 보정합니다.

    1순위: 상품분류(소) 정확 매칭
      - 상품분류(대), 상품분류(중) 수정

    2순위: 상품분류(소) 괄호 안 텍스트 매칭
      - 상품분류(대), 상품분류(중) 수정

    3순위: 위 2개가 실패한 경우 상품분류(중) 매칭
      - 상품분류(대)만 수정
      - 단, 주문제작 상품은 제외
    """
    need_cols = ["상품분류(대)", "상품분류(중)", "상품분류(소)"]

    for col in need_cols:
        if col not in df.columns:
            raise ValueError(f"결과 데이터에 '{col}' 컬럼이 없습니다.")

    cat = pd.read_excel(category_path, dtype=str, keep_default_na=False)

    for col in need_cols:
        if col not in cat.columns:
            raise ValueError(f"카테고리 파일에 '{col}' 컬럼이 없습니다.")

    work_df = df.copy()

    for col in need_cols:
        work_df[col] = work_df[col].apply(clean_category_text)
        cat[col] = cat[col].apply(clean_category_text)

    # =========================
    # 1. 상품분류(소) 기준 매칭표
    # =========================
    cat["_small_key"] = cat["상품분류(소)"].apply(make_category_key)

    cat_small_map = (
        cat.dropna(subset=["_small_key"])
           .drop_duplicates(subset=["_small_key"], keep="first")
           .set_index("_small_key")[["상품분류(대)", "상품분류(중)"]]
    )

    work_df["_상품분류_key_exact"] = work_df["상품분류(소)"].apply(make_category_key)
    work_df["_상품분류_key_parenthesis"] = work_df["상품분류(소)"].apply(get_parenthesis_category_key)

    exact_mask = work_df["_상품분류_key_exact"].isin(cat_small_map.index)
    parenthesis_mask = (~exact_mask) & work_df["_상품분류_key_parenthesis"].isin(cat_small_map.index)

    work_df["_상품분류_match_key"] = pd.NA
    work_df.loc[exact_mask, "_상품분류_match_key"] = work_df.loc[exact_mask, "_상품분류_key_exact"]
    work_df.loc[parenthesis_mask, "_상품분류_match_key"] = work_df.loc[parenthesis_mask, "_상품분류_key_parenthesis"]

    small_match_mask = work_df["_상품분류_match_key"].notna()

    # 상품분류(소) 매칭 성공 시 상품분류(대), 상품분류(중) 수정
    work_df.loc[small_match_mask, "상품분류(대)"] = (
        work_df.loc[small_match_mask, "_상품분류_match_key"].map(cat_small_map["상품분류(대)"])
    )
    work_df.loc[small_match_mask, "상품분류(중)"] = (
        work_df.loc[small_match_mask, "_상품분류_match_key"].map(cat_small_map["상품분류(중)"])
    )

    # =========================
    # 2. 상품분류(중) 기준 상품분류(대) 보정
    # - 상품분류(소) 매칭 실패한 행만 대상
    # - 주문제작 상품은 제외
    # =========================
    cat["_mid_key"] = cat["상품분류(중)"].apply(make_category_key)

    cat_mid_map = (
        cat.dropna(subset=["_mid_key"])
           .drop_duplicates(subset=["_mid_key"], keep="first")
           .set_index("_mid_key")["상품분류(대)"]
    )

    work_df["_상품분류_mid_key"] = work_df["상품분류(중)"].apply(make_category_key)

    category_text_for_exclude = (
        work_df[need_cols]
        .fillna("")
        .astype(str)
        .agg(" ".join, axis=1)
    )

    주문제작_제외_mask = category_text_for_exclude.str.contains("주문제작", na=False)

    mid_fallback_mask = (
        (~small_match_mask)
        & (~주문제작_제외_mask)
        & work_df["_상품분류_mid_key"].isin(cat_mid_map.index)
    )

    # 상품분류(중) 매칭은 상품분류(대)만 수정
    work_df.loc[mid_fallback_mask, "상품분류(대)"] = (
        work_df.loc[mid_fallback_mask, "_상품분류_mid_key"].map(cat_mid_map)
    )

    # 최종 매칭 여부
    final_match_mask = small_match_mask | mid_fallback_mask

    # =========================
    # 3. 요약 / 미매칭 정리
    # =========================
    count_big = (
        work_df["상품분류(대)"]
        .fillna("(빈값)")
        .value_counts()
        .reset_index()
    )
    count_big.columns = ["상품분류(대)", "개수"]

    count_mid = (
        work_df["상품분류(중)"]
        .fillna("(빈값)")
        .value_counts()
        .reset_index()
    )
    count_mid.columns = ["상품분류(중)", "개수"]

    count_small = (
        work_df["상품분류(소)"]
        .fillna("(빈값)")
        .value_counts()
        .reset_index()
    )
    count_small.columns = ["상품분류(소)", "개수"]

    category_unmatched = (
        work_df.loc[~final_match_mask, ["상품분류(대)", "상품분류(중)", "상품분류(소)"]]
        .fillna("(빈값)")
        .value_counts()
        .reset_index(name="개수")
    )

    category_summary = pd.DataFrame({
        "항목": [
            "전체 행수",
            "상품분류(소) 정확 매칭",
            "상품분류(소) 괄호 안 추가 매칭",
            "상품분류(중) 기준 대분류 보정",
            "주문제작 제외 행수",
            "상품분류 총 매칭",
            "상품분류 미매칭",
            "상품분류(소) 빈값"
        ],
        "개수": [
            len(work_df),
            int(exact_mask.sum()),
            int(parenthesis_mask.sum()),
            int(mid_fallback_mask.sum()),
            int(주문제작_제외_mask.sum()),
            int(final_match_mask.sum()),
            int((~final_match_mask).sum()),
            int(work_df["상품분류(소)"].isna().sum())
        ]
    })

    drop_cols = [
        "_상품분류_key_exact",
        "_상품분류_key_parenthesis",
        "_상품분류_match_key",
        "_상품분류_mid_key"
    ]
    work_df = work_df.drop(columns=[c for c in drop_cols if c in work_df.columns])

    category_count_dfs = {
        "상품분류_대분류개수": count_big,
        "상품분류_중분류개수": count_mid,
        "상품분류_소분류개수": count_small,
    }

    return work_df, category_summary, category_unmatched, category_count_dfs


log("상품분류 보정 함수 준비 완료: 소분류 실패 시 중분류 기준 대분류 보정 포함 / 주문제작 제외")


## 7. 상품분류 보정 및 저장


In [ ]:
try:
    import xlsxwriter  # noqa: F401
    EXCEL_ENGINE = "xlsxwriter"
    ENGINE_KWARGS = {"options": {"strings_to_urls": False}}
except ImportError:
    EXCEL_ENGINE = "openpyxl"
    ENGINE_KWARGS = {}

log("Excel 저장 엔진:", EXCEL_ENGINE)

# ------------------------------------------------------------
# 상품분류(대)/(중) 보정
# ------------------------------------------------------------
# 구매처명 매칭이 끝난 result_df에 상품분류 수정까지 함께 반영합니다.
result_df, category_summary, category_unmatched, category_count_dfs = apply_product_category_fix(
    result_df,
    FILE_CATEGORY
)

# 미매칭 판촉사랑 시트도 보정된 result_df 기준으로 다시 생성합니다.
unmatched_87_df = result_df.loc[unmatched_87_indexes].copy() if unmatched_87_indexes else pd.DataFrame(columns=result_df.columns)

with pd.ExcelWriter(OUTPUT_FILE, engine=EXCEL_ENGINE, engine_kwargs=ENGINE_KWARGS) as writer:
    result_df.to_excel(writer, sheet_name=f"{SOURCE_LABEL}_구매처추가_1대1", index=False)
    matched_detail_df.to_excel(writer, sheet_name="매칭상세_거래데이터원본", index=False)
    unmatched_87_df.to_excel(writer, sheet_name=f"미매칭_{SOURCE_LABEL}", index=False)
    unused_trade_df.to_excel(writer, sheet_name="미사용_거래데이터원본", index=False)
    review_df.to_excel(writer, sheet_name="후보다수_검토", index=False)
    summary.to_excel(writer, sheet_name="요약", index=False)

    # 상품분류 보정 결과 확인용 시트
    category_summary.to_excel(writer, sheet_name="상품분류_수정요약", index=False)
    category_unmatched.to_excel(writer, sheet_name="상품분류_미매칭", index=False)
    category_count_dfs["상품분류_대분류개수"].to_excel(writer, sheet_name="상품분류_대분류개수", index=False)
    category_count_dfs["상품분류_중분류개수"].to_excel(writer, sheet_name="상품분류_중분류개수", index=False)
    category_count_dfs["상품분류_소분류개수"].to_excel(writer, sheet_name="상품분류_소분류개수", index=False)

log("상품분류 수정 반영 완료:", FILE_CATEGORY)
log("저장 완료:", OUTPUT_FILE)
log("\n상품분류 수정 요약:")
log(category_summary.to_string(index=False))


## 8. 검증


In [ ]:
log("결과 파일 존재:", OUTPUT_FILE.exists())
log("전체 행 수 유지:", len(result_df), "=", len(df87))
log("사용된 거래데이터 행 수:", len(used_trade_indexes))
log("중복 사용된 거래데이터 index 수:", len(used_trade_indexes) - len(set(used_trade_indexes)))

# 예시 확인: 태권도/운동학원 물티슈 케이스가 있으면 표시
sample_mask = result_df[COL_87_PRODUCT].map(clean_text).str.contains("화이트\(백색\) 10매 물티슈", regex=True, na=False)
if sample_mask.any():
    cols = [c for c in [COL_87_KEY, COL_87_CASE_ID, COL_87_PRODUCT, COL_87_DATE, COL_87_MID, COL_87_SMALL, "구매처분류(대)", "구매처명", "매칭상태", "상품명매칭방식", "BERT유사도", "매칭건수"] if c]
    display(result_df.loc[sample_mask, cols].head(20))

# 예시 확인: 코리아5링 니들3색 터치펜 / 중.고등학교 케이스가 있으면 표시
sample_mask2 = result_df[COL_87_PRODUCT].map(clean_text).str.contains("코리아5링 니들3색 터치펜", regex=False, na=False)
if sample_mask2.any():
    cols = [c for c in [COL_87_KEY, COL_87_CASE_ID, COL_87_PRODUCT, COL_87_DATE, COL_87_MID, COL_87_SMALL, "구매처분류(대)", "구매처명", "매칭상태", "상품명매칭방식", "BERT유사도", "매칭건수"] if c]
    display(result_df.loc[sample_mask2, cols].head(20))

if not matched_detail_df.empty:
    display(matched_detail_df.head(20))
# 예시 확인: 포켓에코백 A형 사이즈 표기 보정 케이스가 있으면 표시
sample_mask3 = result_df[COL_87_PRODUCT].map(clean_text).str.contains("포켓에코백 A형", regex=False, na=False)
if sample_mask3.any():
    cols = [c for c in [COL_87_KEY, COL_87_CASE_ID, COL_87_PRODUCT, COL_87_DATE, COL_87_MID, COL_87_SMALL, "구매처분류(대)", "구매처명", "매칭상태", "상품명매칭방식", "BERT유사도", "매칭건수"] if c]
    display(result_df.loc[sample_mask3, cols].head(20))
